In [1]:

import numpy as np

from scripts.data_filter import filter_dataframe
from scripts.data_loader import load_dataframe
from scripts.utils import RF_PARAM_5G, NETWORK_TYPE

filename = "5G_data_2023.mat"

# Series of random seeds for reproducability
random_seeds = np.loadtxt("../data/random_seeds.csv", dtype=int)

# load the dataframe from saved file or 'raw' matlab file
df = load_dataframe(filename, NETWORK_TYPE._5G)

# # Drop unused columns to save space
matrix_cols_to_drop = ["toa_pps", "toa_cir", "toa_cov", "campaign_id"]
df["measurements_matrix"] = df["measurements_matrix"].apply(
    lambda x: x.drop(columns=matrix_cols_to_drop)
)

operator_choice = [10]
selected_campaigns = list(range(1, 21))
rf_param = RF_PARAM_5G.RSRQ

# Data filtering
df_orig = filter_dataframe(
    df=df,
    operators=operator_choice,
    include_columns=[
        "pci",
        "beam_index",
        "nr_arfcn",
        "operator_id",
        "sinr",
        "rsrq"
    ],
    campaigns=selected_campaigns,
)


Loaded dataframe from .h5 file: /Users/andreres/Documents/UIO/master/thesis/dev/5G_localization/data/dataframe_cache/5G_data_2023.h5


/Users/andreres/Documents/UIO/master/thesis/dev/5G_localization/scripts/data_filter.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["measurements_matrix"] = df["measurements_matrix"].apply(


In [9]:
from scripts.beamforming import get_best_beam, filter_best_beams
from scripts.utils import dataset_tp_rp_split, extract_unique_npcis
from scripts.weighted_coverage import wknn_one
from scripts.matrix_operations import compute_weights, create_point_matrix
import pandas as pd

df = df_orig.sample(200)

df["best_beam"] = df["measurements_matrix"].apply(
    lambda x: get_best_beam(x, rf_param)
)

df_tp, df_rp = dataset_tp_rp_split(df, 0.3, 12354)

unique_pcis = extract_unique_npcis(df['measurements_matrix'])

m_rp, idx_rp = create_point_matrix(df_rp, unique_pcis, rf_param)

errors = []
for _, row in df.iterrows():
    tp = pd.DataFrame([row])

    tp.loc[:, "measurements_matrix"] = tp.loc[:, "measurements_matrix"].apply(
        lambda x: filter_best_beams(x, rf_param, n_best_pcis=1, use_sidelobes=False)
    )

    m_tp, idx_tp = create_point_matrix(tp, unique_pcis, rf_param)
    W, idx_sort = compute_weights(m_rp, idx_rp, m_tp, idx_tp)
    _, err = wknn_one(tp, df_rp, idx_sort, W, 1)

    errors.extend(err)

errors = np.array(errors)
print(errors)
print(f'MEAN {errors.mean():.2f}')



[114.15224824  15.34166077  39.73910258 123.97469156  48.97725458
   8.58366607   4.91473162  16.58611477  82.84765908 159.86643419
 263.66245835 228.93179016 111.69133799  71.64351754  88.03112748
  19.8306482  174.57959312 127.53382755  59.30048928  43.19327014
 237.0013106  182.37572     32.86091438   2.56283469  61.95194011
 220.22163887  42.83377262 104.03102163 220.43950661 141.70873223
 184.24878887  17.73305553  19.18338893 139.98710404 142.67954474
   1.24645703  71.53071621 146.89425344 108.31988648   6.73956189
  15.39377506 130.67812807 251.48265309  47.7218019  123.61585167
 116.2457384  140.14606533 142.8432298   30.2835332  128.17531085
 157.74173509 127.4468734   35.47758495  39.01125451 145.92941545
  41.84865104  13.44880061  27.93869772  50.47034305  99.25344685
 136.17090398  53.00892313  46.66589293 126.56384508  35.01908731
  24.17290114 128.40090606  72.22677372  49.4648919  190.74259394
 133.18331221  63.58520205   0.         133.15897933 207.47501975
  96.41571

In [3]:
res

NameError: name 'res' is not defined